# VectorForge OCR CUDA Benchmark - Google Colab

This notebook runs the VectorForge Part 2 OCR benchmark on a Colab NVIDIA GPU. It follows the same workflow used in the project code: verify CUDA, clone the repo, install missing dependencies without replacing Colab's GPU-enabled PyTorch build, run tests, generate the synthetic OCR dataset, train on CUDA, benchmark, and inspect saved results.

## 1. Enable the Colab GPU

Before running the notebook, choose **Runtime > Change runtime type > Hardware accelerator > T4 GPU**. Then run this cell and confirm `CUDA available` is `True`.

In [ ]:
import subprocess
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

if torch.cuda.is_available():
    subprocess.run(["nvidia-smi"], check=False)


## 2. Clone VectorForge

If the repo is already present in `/content`, this cell will reuse it. Otherwise it clones the GitHub copy.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/AbuWabu3697/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine.git"
PROJECT_DIR = Path("/content/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

%cd /content/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine
!ls


## 3. Install Project Dependencies

This intentionally does not reinstall `torch`; Colab usually provides a CUDA-enabled PyTorch build already, and replacing it with the wrong wheel can make CUDA disappear.

In [ ]:
%pip install -q datasets faiss-cpu matplotlib numpy pandas Pillow pyarrow pytest PyYAML sentence-transformers tqdm


## 4. Run the Test Suite

On a working Colab GPU runtime, the CUDA-specific OCR test should run instead of being skipped.

In [ ]:
!python -m pytest tests -q -p no:cacheprovider


## 5. Generate the Synthetic OCR Dataset

The trainer expects `data/ocr/metadata/samples.csv`, so generate the OCR samples before training.

In [ ]:
!python -m src.ocr.data.synthetic_generator --config config/ocr.yaml


## 6. Train CRNN + CTC on CUDA

This writes training history, metadata, and checkpoints under `results/ocr/`.

In [ ]:
!python -m src.ocr.training.trainer --config config/ocr.yaml --device cuda --precision fp32


## 7. Run the Baseline CPU/GPU Benchmark

The baseline experiment records CPU and CUDA throughput using the shared benchmark runner.

In [ ]:
!python -m src.ocr.benchmarks.ocr_benchmark --config config/ocr.yaml --experiment baseline


## 8. List Saved Artifacts

In [ ]:
!find results/ocr -maxdepth 3 -type f | sort


## 9. Preview CSV Results

In [ ]:
!python - <<'PY'
import pandas as pd
from pathlib import Path

for path in sorted(Path("results/ocr").rglob("*.csv")):
    print(f"
=== {path} ===")
    print(pd.read_csv(path).to_string(index=False))
PY


## 10. Generate and Display Plots

In [ ]:
!python -m src.visualization.ocr_plots

from IPython.display import Image, display
from pathlib import Path

for path in sorted(Path("results/ocr").glob("*.png")):
    print(path)
    display(Image(filename=str(path)))


## 11. Optional: Download Results

Run this after training or benchmarking if you want to keep the Colab artifacts before the runtime resets.

In [ ]:
!zip -qr vectorforge_ocr_results.zip results/ocr

from google.colab import files
files.download("vectorforge_ocr_results.zip")
